In [ ]:
# 

## Load packages

In [ ]:
import scipy.sparse as sp
import numpy as np
import pandas as pd
import scanpy as sc

## load data

In [ ]:
adata_hgv = sc.read('/storage/users/data/PANC/H5AD_file/adata_filtered_no2D_hvg_clust_time4_clust_t-bin.h5ad')

In [ ]:
adata_full = sc.read('/storage/users/data/PANC/H5AD_file/adata_filtered_no2D.h5ad')

In [ ]:
adata_hgv

In [ ]:
adata_full 

## merge clusters with all genes

In [ ]:
###############################################
# 1.  Make sure the barcodes match exactly
###############################################
# If barcodes already carry sample prefixes etc., drop them here as needed.
common_barcodes = adata_hgv.obs_names.intersection(adata_full.obs_names)
print(f"{len(common_barcodes):,} cells present in BOTH objects")

# (Optional sanity-check for lost cells)
lost_hgv   = adata_hgv.n_obs - len(common_barcodes)
lost_full  = adata_full.n_obs - len(common_barcodes)
print(f"  – {lost_hgv} barcodes found only in adata_hgv")
print(f"  – {lost_full} barcodes found only in adata_full")

###############################################
# 2.  Subset the *full* matrix to those cells
###############################################
adata_full_sync = adata_full[common_barcodes, :].copy()

###############################################
# 3.  Transfer (or merge) the obs-level annotations
###############################################
cols_to_copy = [
    "leiden_t_bin_merged_nicer",   # main grouping
    # plus any others you might use later:
    "leiden", "t", "milestones", "condition"
]

for col in cols_to_copy:
    adata_full_sync.obs[col] = adata_hgv.obs.loc[adata_full_sync.obs_names, col]

# If you want all OBS columns verbatim:
# adata_full_sync.obs = adata_hgv.obs.loc[adata_full_sync.obs_names]

###############################################
# 4.  Bring over the colour dictionary (uns) if needed
###############################################
key = "leiden_t_bin_merged_nicer_colors"
if key in adata_hgv.uns:
    adata_full_sync.uns[key] = adata_hgv.uns[key]

###############################################
# 5.  Confirm it worked
###############################################
print(adata_full_sync)
print(adata_full_sync.obs.head())


In [ ]:
adata_full_sync

In [ ]:
adata_full_sync.write_h5ad("/storage/users/data/PANC/H5AD_file/adata_full_sync.h5ad")

## Extract nodes

In [ ]:
############################################################
# 0.  Inventory
############################################################
print(f"Cells in adata_full_sync : {adata_full_sync.n_obs:,}")
print(f"Genes in adata_full_sync : {adata_full_sync.n_vars:,}")

############################################################
# 1.  Mean, median, SD per leiden-t bin
############################################################
import scipy.sparse as sp
import numpy as np
import pandas as pd

group_key  = "leiden_t_bin_merged_nicer"
groups     = adata_full_sync.obs[group_key].unique().tolist()
gene_names = adata_full_sync.var_names.tolist()

expr_mean   = pd.DataFrame(index=gene_names, columns=groups, dtype=np.float32)
expr_median = pd.DataFrame(index=gene_names, columns=groups, dtype=np.float32)
expr_sd     = pd.DataFrame(index=gene_names, columns=groups, dtype=np.float32)   # optional

for g in groups:
    mask = adata_full_sync.obs[group_key] == g
    Xg   = adata_full_sync[mask].X                    # sparse matrix view

    # ----- convert to dense only once -----
    if sp.issparse(Xg):
        Xg = Xg.toarray()                             # <- was  Xg.A

    expr_mean[g]   = Xg.mean(axis=0).ravel()
    expr_median[g] = np.median(Xg, axis=0)
    expr_sd[g]     = Xg.std(axis=0, ddof=1)

print(f"Computed mean/median for {len(groups)} groups")

############################################################
# 2.  Detection cut-off  (10th percentile of non-zero values)
############################################################
vals    = adata_full_sync.X.data if sp.issparse(adata_full_sync.X) else adata_full_sync.X.flatten()
cutoff  = np.quantile(vals[vals > 0], 0.05)
print(f"Chosen cut-off: {cutoff:.4f}")

############################################################
# 3.  Filter genes that never exceed the cut-off
############################################################
keep_mask   = (expr_mean > cutoff).any(axis=1)
expr_mean   = expr_mean  .loc[keep_mask]
expr_median = expr_median.loc[keep_mask]
expr_sd     = expr_sd    .loc[keep_mask]     # optional

print(f"Genes kept (> cut-off in ≥1 group): {keep_mask.sum():,} of {len(keep_mask):,}")
print("\nCells per group:")
print(adata_full_sync.obs[group_key].value_counts().sort_index())

############################################################
# 4.  Persist results
############################################################
expr_mean  .to_csv("expr_mean_by_group.csv")
expr_median.to_csv("expr_median_by_group.csv")
expr_sd    .to_csv("expr_sd_by_group.csv")   # optional


In [ ]:
import scipy.sparse as sp
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1.  Collect every expression value and keep only the non-zeros
# ------------------------------------------------------------------
vals = (adata_full_sync.X.data
        if sp.issparse(adata_full_sync.X)
        else adata_full_sync.X.flatten())
nonzero = vals[vals > 0]

# ------------------------------------------------------------------
# 2.  Reproduce the cut-off (10-th percentile of non-zeros)
# ------------------------------------------------------------------
cutoff = np.quantile(nonzero, 0.10)

# ------------------------------------------------------------------
# 3.  Visualise
# ------------------------------------------------------------------
plt.figure(figsize=(6, 4.5))
plt.hist(nonzero, bins=100)
plt.axvline(cutoff, linestyle="--")        # cut-off marker
plt.yscale("log")                          # long right tail → log-scale y-axis
plt.xlabel("log1p(counts)")
plt.ylabel("Frequency (log scale)")
plt.title("Distribution of non-zero expression values\nwith 10th-percentile cut-off")
plt.tight_layout()
plt.show()


In [ ]:
import scipy.sparse as sp
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1.  Collect every expression value and keep only the non-zeros
# ------------------------------------------------------------------
vals = (adata_hgv.X.data
        if sp.issparse(adata_hgv.X)
        else adata_hgv.X.flatten())
nonzero = vals[vals > 0]

# ------------------------------------------------------------------
# 2.  Reproduce the cut-off (10-th percentile of non-zeros)
# ------------------------------------------------------------------
cutoff = np.quantile(nonzero, 0.10)

# ------------------------------------------------------------------
# 3.  Visualise
# ------------------------------------------------------------------
plt.figure(figsize=(6, 4.5))
plt.hist(nonzero, bins=100)
plt.axvline(cutoff, linestyle="--")        # cut-off marker
plt.yscale("log")                          # long right tail → log-scale y-axis
plt.xlabel("log1p(counts)")
plt.ylabel("Frequency (log scale)")
plt.title("Distribution of non-zero expression values\nwith 10th-percentile cut-off")
plt.tight_layout()
plt.show()


In [ ]:
import anndata as ad
import pandas as pd

############################################################
# 0. Build a cells-per-group table
############################################################
cells_per_group = (
    adata_full_sync
    .obs["leiden_t_bin_merged_nicer"]
    .value_counts()
)

############################################################
# 1. Create adata_mean:
#    – obs: one row per leiden group, with n_cells + group label
#    – var: genes that survived the cut-off
#    – X:   mean log-expression (groups × genes)
############################################################
obs_df = pd.DataFrame({
    "n_cells": cells_per_group.loc[expr_mean.columns].values,
    "leiden_t_bin_merged_nicer": expr_mean.columns
}, index=expr_mean.columns)

adata_mean = ad.AnnData(
    X   = expr_mean.T.astype("float32"),  # groups × genes = mean
    obs = obs_df,
    var = pd.DataFrame(index=expr_mean.index)
)

# Add layers: mean (optional, since X already is mean) and SD
adata_mean.layers["mean"] = expr_mean.T.astype("float32")
adata_mean.layers["sd"]   = expr_sd.T.astype("float32")

############################################################
# 2. Quick report
############################################################
print("New AnnData object:")
print(adata_mean)  
print("\nFirst few obs rows:")
print(adata_mean.obs.head(25))

############################################################
# 3. Save if you like
############################################################
adata_mean.write_h5ad("expr_mean_by_group.h5ad")


In [ ]:
adata_full

In [ ]:
adata_mean

In [ ]:
adata_mean.write_h5ad("/storage/users/data/PANC/H5AD_file/adata_expr_mean_by_group.h5ad")